# TCD-TIMIT Preprocessing (Clean)

This notebook is intentionally TCD-TIMIT only:
- Convert videos to 25 fps
- Convert audio to 16 kHz mono
- Extract face landmarks
- Create aligned mouth crops


In [ ]:
from pathlib import Path
import os
import subprocess

# ===== User config =====
DATA_ROOT = Path(r"D:\TCD_TIMIT")
GROUP = "volunteers"  # volunteers | lipspeakers
SPEAKER = "01M"       # set None to process all speakers in GROUP
CAM = "straightcam"    # straightcam | 30degcam

FFMPEG_EXE = "ffmpeg"
PYTHON_EXE = os.environ.get("PYTHON_EXECUTABLE", "python")

DETECT_LANDMARK_PY = Path(r"C:\Users\irish\Computer_Electronic_Engineering_Year5\AVSR_project\av_hubert\av_hubert\avhubert\preparation\detect_landmark.py")
ALIGN_MOUTH_PY = Path(r"C:\Users\irish\Computer_Electronic_Engineering_Year5\AVSR_project\av_hubert\av_hubert\avhubert\preparation\align_mouth_stabilised.py")
MEAN_FACE_NPY = Path(r"C:\Users\irish\Computer_Electronic_Engineering_Year5\AVSR_project\assets\mean_face\20words_mean_face.npy")
DLIB_CNN = Path(r"C:\models\dlib\mmod_human_face_detector.dat")
DLIB_L68 = Path(r"C:\models\dlib\shape_predictor_68_face_landmarks.dat")

assert DATA_ROOT.exists(), f"Missing DATA_ROOT: {DATA_ROOT}"
assert DETECT_LANDMARK_PY.exists(), f"Missing detect_landmark.py: {DETECT_LANDMARK_PY}"
assert ALIGN_MOUTH_PY.exists(), f"Missing align_mouth_stabilised.py: {ALIGN_MOUTH_PY}"
assert MEAN_FACE_NPY.exists(), f"Missing mean face: {MEAN_FACE_NPY}"
assert DLIB_CNN.exists(), f"Missing dlib CNN model: {DLIB_CNN}"
assert DLIB_L68.exists(), f"Missing dlib 68 model: {DLIB_L68}"
print("Config loaded.")


In [ ]:
def run_cmd(cmd: list):
    cmd = [str(c) for c in cmd]
    print(">>", " ".join(cmd))
    subprocess.run(cmd, check=True)

def speaker_dirs(root: Path, group: str, speaker: str | None):
    base = root / group
    if speaker:
        p = base / speaker
        assert p.exists(), f"Speaker not found: {p}"
        return [p]
    return sorted([p for p in base.iterdir() if p.is_dir()])

def convert_media_for_cam(speaker_dir: Path, cam: str):
    clips = speaker_dir / "Clips"
    in_vid = clips / "video" / cam
    in_wav = clips / "audio" / cam
    out_vid = clips / "processed" / "video25" / cam
    out_wav = clips / "processed" / "audio16k" / cam
    out_vid.mkdir(parents=True, exist_ok=True)
    out_wav.mkdir(parents=True, exist_ok=True)

    v_in = sorted(in_vid.glob("*.mp4"))
    a_in = sorted(in_wav.glob("*.wav"))

    for src in v_in:
        dst = out_vid / src.name
        run_cmd([FFMPEG_EXE, "-y", "-v", "error", "-i", src, "-r", "25", "-an", "-c:v", "libx264", "-preset", "veryfast", "-crf", "18", dst])

    for src in a_in:
        dst = out_wav / src.name
        run_cmd([FFMPEG_EXE, "-y", "-v", "error", "-i", src, "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", dst])

    print(f"{speaker_dir.name}/{cam}: converted {len(v_in)} videos and {len(a_in)} audios")


In [ ]:
targets = speaker_dirs(DATA_ROOT, GROUP, SPEAKER)
for spk in targets:
    convert_media_for_cam(spk, CAM)
print("Done: 25fps + 16kHz conversion")


In [ ]:
def build_manifest(speaker_dir: Path, cam: str):
    clips = speaker_dir / "Clips"
    vid25 = clips / "processed" / "video25" / cam
    lmk = clips / "processed" / "landmarks" / cam
    lmk.mkdir(parents=True, exist_ok=True)
    stems = sorted([p.stem for p in vid25.glob("*.mp4")])
    manifest = lmk / "file.list"
    manifest.write_text("\n".join(stems) + ("\n" if stems else ""), encoding="utf-8")
    return vid25, lmk, manifest, stems

for spk in targets:
    vid25, lmk, manifest, stems = build_manifest(spk, CAM)
    print(f"{spk.name}/{CAM}: {len(stems)} clips in manifest")


In [ ]:
for spk in targets:
    vid25, lmk, manifest, stems = build_manifest(spk, CAM)
    if not stems:
        print(f"{spk.name}/{CAM}: no videos found, skipping landmarks")
        continue
    run_cmd([
        PYTHON_EXE, DETECT_LANDMARK_PY,
        "--video-direc", vid25,
        "--filename-path", manifest,
        "--landmark-path", lmk,
        "--cnn_detector", DLIB_CNN,
        "--face_predictor", DLIB_L68,
    ])
print("Done: landmark extraction")


In [ ]:
for spk in targets:
    clips = spk / "Clips"
    vid25 = clips / "processed" / "video25" / CAM
    lmk = clips / "processed" / "landmarks" / CAM
    out_roi = clips / "processed" / "video25crop_alignmouth" / CAM
    out_roi.mkdir(parents=True, exist_ok=True)

    stems = sorted([p.stem for p in vid25.glob("*.mp4") if (lmk / f"{p.stem}.pkl").exists()])
    if not stems:
        print(f"{spk.name}/{CAM}: no video+landmark pairs, skipping crop")
        continue

    filelist = out_roi / "file.list"
    filelist.write_text("\n".join(stems) + "\n", encoding="utf-8")

    run_cmd([
        PYTHON_EXE, ALIGN_MOUTH_PY,
        "--video-direc", vid25,
        "--landmark-direc", lmk,
        "--filename-path", filelist,
        "--save-direc", out_roi,
        "--mean-face", MEAN_FACE_NPY,
        "--crop-width", "96",
        "--crop-height", "96",
        "--start-idx", "48",
        "--stop-idx", "68",
        "--window-margin", "12",
        "--ffmpeg", FFMPEG_EXE,
        "--rank", "0",
        "--nshard", "1",
        "--gray", "0",
        "--stab-alpha", "0.15",
        "--mouth-alpha", "0.20",
        "--max-t-jump", "3.0",
        "--max-r-jump", "3.0",
        "--max-s-jump", "0.03",
    ])
    print(f"{spk.name}/{CAM}: mouth crop complete ({len(stems)} clips)")
print("Done: mouth ROI cropping")


## Notes
- Run cells from top to bottom.
- Set `SPEAKER = None` to process all speakers.
- If your dlib model paths differ, update them in Cell 2.
